In [1]:
import pandas as pd
from fsspec.utils import tokenize
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
import torch
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
import numpy as np
from torch.nn import CrossEntropyLoss
from transformers import get_linear_schedule_with_warmup
import warnings
warnings.filterwarnings("ignore")

c:\Users\acer\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dev_df=pd.read_csv("/mnt/d/EmoBase/Data/dev.txt",sep="\t")
test_df=pd.read_csv("/mnt/d/EmoBase/Data/test.txt",sep="\t")
train_df=pd.read_csv("/mnt/d/EmoBase/Data/train.txt",sep="\t")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/d/EmoBase/Data/dev.txt'

In [3]:
train_df.isnull().sum()
dev_df.isnull().sum()
test_df.isnull().sum()
test_df.dropna()
test_df.reset_index(drop=True)
train_df.reset_index(drop=True)
dev_df.reset_index(drop=True)

,id,turn1,turn2,turn3,label
0,0,Then dont ask me,YOURE A GUY NOT AS IF YOU WOULD UNDERSTAND,IM NOT A GUY FUCK OFF,angry
1,1,Mixed things such as??,the things you do.,Have you seen minions??,others
2,2,Today I'm very happy,and I'm happy for you ❤,I will be marry,happy
3,3,Woah bring me some,left it there oops,Brb,others
4,4,it is thooooo,I said soon master.,he is pressuring me,others
...,...,...,...,...,...
2750,2750,U are my book,book for what? ugliness? THANK YOU,U like ur self,others
2751,2751,I'll be crying,You just want to make ppl cry:P,ppl,others
2752,2752,Thanks for sending,hahaha you're welcome! 😤😤,Why are u not sending,others
2753,2753,Write it,Mr. F,U understand me?,others


In [4]:
train_df = train_df.dropna()
dev_df = dev_df.dropna()
test_df = test_df.dropna()

In [5]:
train_df["text"]=train_df["turn1"]+" [SEP] "+train_df["turn2"]+" [SEP] "+train_df["turn3"]
test_df["text"]=test_df["turn1"]+" [SEP] "+test_df["turn2"]+" [SEP] "+test_df["turn3"]
dev_df["text"]=dev_df["turn1"]+" [SEP] "+dev_df["turn2"]+" [SEP] "+dev_df["turn3"]

In [6]:
emotion2label = {
    "others": 0,
    "happy": 1,
    "sad": 2,
    "angry": 3
}

train_df["label_id"]=train_df["label"].map(emotion2label)

dev_df["label_id"]=dev_df["label"].map(emotion2label)

train_df["label_id"].value_counts()

dev_df["label_id"].value_counts()

label_id
0    2335
3     150
1     142
2     125
Name: count, dtype: int64

In [7]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=4)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# device = torch.device('cuda')
# model = model.to(device)


encoded_train = tokenizer(
    train_df["text"].tolist(),
    max_length=128,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)


# encoded_train["input_ids"].shape

encoded_dev=tokenizer(dev_df["text"].tolist(),max_length=128,return_tensors="pt",padding="max_length",truncation=True)

encoded_test=tokenizer(test_df["text"].tolist(),max_length=128,return_tensors="pt",padding="max_length",truncation=True)



Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1670.61it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import torch
train_labels=torch.tensor(train_df["label_id"].tolist())
dev_labels=torch.tensor(dev_df["label_id"].tolist())
print(train_labels[:5])

tensor([0, 3, 0, 3, 0])


In [9]:
test_df["label_id"] = test_df["label"].map(emotion2label)
test_labels = torch.tensor(test_df["label_id"].tolist())

In [10]:
from torch.utils.data import Dataset
class EmoDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx])
        }
        return item

In [11]:
test_dataset=EmoDataset(encoded_test,test_labels)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [12]:
train_dataset=EmoDataset(encoded_train,train_labels)
dev_dataset=EmoDataset(encoded_dev,dev_labels)
test_dataset = EmoDataset(encoded_test, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
dev_loader=DataLoader(dev_dataset,batch_size=32,shuffle=False)

y=train_labels.numpy()
class_weight=compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3,]),
    y=y
)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = model.to(device)

Using device: cuda


In [ ]:
from transformers import get_linear_schedule_with_warmup


total_steps = len(train_loader) *` `5
warmup_steps = total_steps // 10


optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weight_tensor)


for epoch in range(7):
    print(f"Epoch {epoch+1}/7")
    model.train()
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

Epoch 1/7
Epoch 2/7
Epoch 3/7
Epoch 4/7


In [ ]:
model.save_pretrained("emocontext_model")
tokenizer.save_pretrained("emocontext_model")

In [ ]:
model.eval()
all_pred=[]
all_labels=[]

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs=model(input_ids=input_ids,attention_mask=attention_mask)
        logits=outputs.logits

        pred=torch.argmax(logits,dim=-1)

        all_pred.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels,all_pred))

In [ ]:
# report = classification_report(all_labels, all_pred)
#
# with open("results.txt", "w") as f:
#     f.write("EmoContext BERT Results\n")
#     f.write("="*40 + "\n")
#     f.write(report)
#
# print("Results saved!")

In [ ]:
# import pandas as pd
# results_df = pd.DataFrame({
#     "true_label": all_labels,
#     "predicted_label": all_pred
# })
# results_df.to_csv("predictions.csv", index=False)

In [ ]:
test_df["label_id"]=test_df["label"].map(emotion2label)


In [ ]:
test_labels=torch.tensor(test_df["label_id"].tolist())

In [ ]:
test_dataset=EmoDataset(encoded_test,test_labels)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [ ]:
model.eval()
all_pred=[]
all_labels=[]
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=-1)
        all_pred.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_pred))

In [ ]:
model.save_pretrained("emocontext_model2")
tokenizer.save_pretrained("emocontext_model2")

In [ ]:
len(test_loader.dataset)

In [ ]:
len(train_loader.dataset)

In [ ]:
# len(train_loader.dataset)
# len(dev_loader.dataset)
len(test_loader.dataset)

In [ ]:
len(train_loader.dataset)
len(dev_loader.dataset)

In [ ]:
len(train_loader.dataset)

In [ ]:
report = classification_report(all_labels, all_pred)

with open("results.txt", "w") as f:
    f.write("EmoContext BERT Results\n")
    f.write("="*40 + "\n")
    f.write(report)

print("Results saved!")
import pandas as pd
results_df = pd.DataFrame({
    "true_label": all_labels,
    "predicted_label": all_pred
})
results_df.to_csv("predictions.csv", index=False)